# 🏥 PrevIA — Assistente de Medicina Preventiva

**PrevIA** é um sistema agêntico de medicina preventiva baseado em RAG (*Retrieval-Augmented Generation*) que consulta diretrizes médicas públicas para responder perguntas sobre prevenção e gerar planos de saúde personalizados.


## O que o PrevIA faz

O sistema recebe uma pergunta ou perfil do usuário e executa automaticamente um pipeline de agentes para gerar respostas fundamentadas em documentos oficiais do Ministério da Saúde, INCA e OMS.

**Duas funcionalidades principais:**

- **Q&A preventivo** — responde perguntas como *"A partir de qual idade devo fazer mamografia?"* ou *"Com que frequência fazer o Papanicolau?"*, sempre com citação da fonte e página do documento
- **Plano preventivo personalizado** — recebe idade, sexo e histórico familiar e gera automaticamente um plano anual com exames, vacinas e hábitos recomendados




## Como funciona

O sistema é orquestrado pelo **LangGraph** e opera com 6 agentes em sequência:

| Agente | Função |
|--------|--------|
| **Supervisor** | Classifica a intenção: Q&A simples ou geração de plano |
| **Retriever** | Busca os trechos mais relevantes nos 39 PDFs indexados |
| **Writer** | Gera a resposta com citações das fontes |
| **Self-Check** | Verifica se a resposta tem suporte nos documentos (anti-alucinação) |
| **Safety** | Adiciona disclaimer médico obrigatório |
| **Automation** | Gera plano preventivo anual via servidor MCP |

Se o Self-Check reprovar a resposta, o sistema faz uma nova busca automaticamente. Se reprovar duas vezes, recusa a resposta por falta de evidência.


## Base de conhecimento

O corpus é composto por **39 documentos públicos** (~7.050 chunks indexados no FAISS) cobrindo:
vacinação, rastreamento de câncer, doenças crônicas, saúde da mulher, nutrição e estilo de vida.


## Stack

| Componente | Tecnologia |
|------------|-----------|
| LLM | Qwen2.5 7B via Ollama (local, sem API) |
| Embeddings | all-MiniLM-L6-v2 (HuggingFace) |
| Busca vetorial | FAISS |
| Orquestração | LangGraph |
| Interface | Streamlit + ngrok |
| MCP Server | health-checklist (próprio) |


## Limitações importantes

- As respostas são baseadas exclusivamente nos documentos indexados
- O sistema **não faz diagnóstico, não prescreve medicamentos e não substitui consulta médica**
- A qualidade das respostas depende do tamanho do modelo e da cobertura do corpus

## Configurações iniciais - LLM

In [8]:
!pip install -q langchain langchain-community langchain-ollama \
               langchain-huggingface langgraph faiss-cpu \
               sentence-transformers pypdf ragas datasets transformers torch mcp gdown \
               streamlit pyngrok pytesseract pdf2image

!sudo apt-get install -y zstd tesseract-ocr tesseract-ocr-por poppler-utils
!curl -fsSL https://ollama.com/install.sh | sh



Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
tesseract-ocr is already the newest version (4.1.1-2.1build1).
tesseract-ocr-por is already the newest version (1:4.00~git30-7274cfa-1.1).
poppler-utils is already the newest version (22.02.0-2ubuntu0.12).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [16]:
import subprocess
import time
import requests
import os
import shutil
import logging
import sys
import re
import asyncio
import json
import getpass
import pandas as pd, json
import torch
import pytesseract
from typing import TypedDict, List, Optional
from collections import Counter
from PIL import Image
from pdf2image import convert_from_path
from pyngrok import ngrok
import gdown

# LangChain e LangGraph
from langchain_ollama import ChatOllama
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END

# MCP
from mcp.server import Server
from mcp.server.stdio import stdio_server
from mcp import types

# Configurações de ambiente
sys.path.append('/content/previa')
os.makedirs('/content/previa/app', exist_ok=True)
os.makedirs('/content/previa/docs', exist_ok=True)
os.makedirs('/content/previa/faiss_db', exist_ok=True)

print("Imports e dependências do sistema configurados!")

Imports e dependências do sistema configurados!


In [10]:
#Iniciar servidor Ollama

subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

try:
    r = requests.get("http://localhost:11434")
    print("Ollama rodando!")
except:
    print("erro")

Ollama rodando!


In [11]:
# Baixar Qwen2.5 7B

!ollama pull qwen2.5:7b

print('Qwen2.5 7B baixado')


Qwen2.5 7B baixado


In [12]:
llm = ChatOllama(model='qwen2.5:7b', temperature=0.2)

resposta = llm.invoke('O que é medicina preventiva? Responda em português e em 2 frases')
print('Teste')
print(resposta.content)
print('\nQwen2.5 7B funcionando')

Teste
A medicina preventiva é uma abordagem que visa prevenir doenças e promover a saúde através de métodos como exames preventivos, vacinas e hábitos de vida saudáveis. Ela enfatiza a importância de manter o corpo em boas condições para evitar problemas de saúde no futuro.

Qwen2.5 7B funcionando


In [13]:


config_code = '''

BASE_DIR        = "/content/previa"
DOCS_DIR        = f"{BASE_DIR}/docs"
FAISS_DIR       = f"{BASE_DIR}/faiss_db"

# LLM via Ollama local
LLM_MODEL       = "qwen2.5:7b"
OLLAMA_BASE_URL = "http://localhost:11434"
TEMPERATURE     = 0.2

# Embeddings HuggingFace (sem API)
EMBED_MODEL     = "sentence-transformers/all-MiniLM-L6-v2"

# RAG
CHUNK_SIZE      = 400
CHUNK_OVERLAP   = 80
TOP_K           = 5

COLLECTION_NAME = "previa_corpus"
'''

os.makedirs('/content/previa', exist_ok=True)
with open('/content/previa/config.py', 'w') as f:
    f.write(config_code)

print('config.py salvo')

config.py salvo


## Indexar PDFs médicos e RAG

In [14]:
# PDFs com URLs

# Limpa a pasta docs completamente
docs_dir = '/content/previa/docs'
shutil.rmtree(docs_dir, ignore_errors=True)
os.makedirs(docs_dir, exist_ok=True)
print('Pasta docs limpa!\n')

docs = [
    {
        'url': 'https://bvsms.saude.gov.br/bvs/publicacoes/inca/rastreamento_cancer_colo_utero.pdf',
        'arquivo': 'rastreamento_cancer_colo_utero.pdf',
        'descricao': 'Diretrizes Brasileiras - Rastreamento Câncer do Colo do Útero (INCA)'
    },
    {
        'url': 'https://bvsms.saude.gov.br/bvs/publicacoes/parametros_rastreamento_cancer_mama.pdf',
        'arquivo': 'parametros_rastreamento_cancer_mama.pdf',
        'descricao': 'Parâmetros Técnicos - Rastreamento Câncer de Mama (INCA)'
    },
    {
        'url': 'https://bvsms.saude.gov.br/bvs/publicacoes/inca/Beatriz_Kneipp_Controle_canceres_colo_utero_mama.pdf',
        'arquivo': 'controle_canceres_utero_mama.pdf',
        'descricao': 'Controle dos Cânceres do Colo do Útero e de Mama'
    },
    {
        'url': 'https://bvsms.saude.gov.br/bvs/publicacoes/plano_action_reducao_cancer_colo.pdf',
        'arquivo': 'plano_reducao_cancer_colo.pdf',
        'descricao': 'Plano de Ação - Redução do Câncer do Colo do Útero'
    },
    {
        'url': 'https://bvsms.saude.gov.br/bvs/publicacoes/situacao_cancer_br3.pdf',
        'arquivo': 'situacao_cancer_brasil.pdf',
        'descricao': 'A Situação do Câncer no Brasil (INCA/MS)'
    },
    {
        'url': 'https://bvsms.saude.gov.br/bvs/boletim_tematico/cancer_colo_utero_marco_2023.pdf',
        'arquivo': 'boletim_cancer_colo_utero_2023.pdf',
        'descricao': 'Boletim Temático - Prevenção Câncer do Colo do Útero (2023)'
    },
    {
        'url': 'https://bvsms.saude.gov.br/bvs/publicacoes/hipertensao_arterial_sistemica_cab37.pdf',
        'arquivo': 'hipertensao_cab37.pdf',
        'descricao': 'Hipertensão Arterial - CAB 37'
    },
    {
        'url': 'https://bvsms.saude.gov.br/bvs/publicacoes/estrategias_cuidado_pessoa_doenca_cronica_cab35.pdf',
        'arquivo': 'doencas_cronicas_cab35.pdf',
        'descricao': 'Doenças Crônicas - CAB 35'
    },
    {
        'url': 'https://bvsms.saude.gov.br/bvs/publicacoes/caderno_atencao_primaria_29_rastreamento.pdf',
        'arquivo': 'rastreamento_cap29.pdf',
        'descricao': 'Rastreamento - CAP 29 (exames por faixa etária)'
    },
]

print('⏳ Baixando PDFs...\n')
baixados = []

for doc in docs:
    destino = f"/content/previa/docs/{doc['arquivo']}"
    if os.path.exists(destino) and os.path.getsize(destino) > 10_000:
        print(f"  Já existe: {doc['descricao']}")
        baixados.append(destino)
        continue
    print(f"  {doc['descricao']}")
    subprocess.run(['wget', '-q', '--timeout=60', '-O', destino, doc['url']], capture_output=True)
    tamanho = os.path.getsize(destino) if os.path.exists(destino) else 0
    if tamanho > 10_000:
        print(f"     OK ({tamanho // 1024} KB)")
        baixados.append(destino)
    else:
        print(f"    Falhou ({tamanho} bytes)")
        if os.path.exists(destino): os.remove(destino)

print(f'\n{len(baixados)}/{len(docs)} PDFs baixados!')

Pasta docs limpa!

⏳ Baixando PDFs...

  Diretrizes Brasileiras - Rastreamento Câncer do Colo do Útero (INCA)
     OK (2414 KB)
  Parâmetros Técnicos - Rastreamento Câncer de Mama (INCA)
     OK (156 KB)
  Controle dos Cânceres do Colo do Útero e de Mama
     OK (1865 KB)
  Plano de Ação - Redução do Câncer do Colo do Útero
    Falhou (0 bytes)
  A Situação do Câncer no Brasil (INCA/MS)
     OK (3625 KB)
  Boletim Temático - Prevenção Câncer do Colo do Útero (2023)
     OK (6885 KB)
  Hipertensão Arterial - CAB 37
     OK (8434 KB)
  Doenças Crônicas - CAB 35
     OK (6170 KB)
  Rastreamento - CAP 29 (exames por faixa etária)
     OK (8245 KB)

8/9 PDFs baixados!


In [17]:
# Baixar PDFs da pasta pública do Google Drive

FOLDER_ID = '1GSeWTNzZv7QGH67aDck_ukE4u7qbvzj4'

print('⏳ Baixando PDFs do Google Drive...')
gdown.download_folder(
    id=FOLDER_ID,
    output='/content/previa/docs',
    quiet=False,
    use_cookies=False
)

# Lista o que foi baixado
pdfs = [f for f in os.listdir('/content/previa/docs') if f.endswith('.pdf')]
print(f'\n{len(pdfs)} PDFs baixados:')
for pdf in sorted(pdfs):
    tam = os.path.getsize(f'/content/previa/docs/{pdf}') // 1024
    print(f'  {pdf} ({tam} KB)')

⏳ Baixando PDFs do Google Drive...


Retrieving folder contents


Processing file 1PrmCOakvFOUrJeGxfRvSYIjvSxAyXogJ 2197-UN-CARTILHA-CUIDADOS COM MEDICAMENTOS V3_compressed.pdf
Processing file 1-Kk6XOBpeYoitB9Y5hsm_6igjTR_Yy5x caderno_tematico_pse_doencas_negligenciadas.pdf
Processing file 1O9VjOYc1s8tsPlWzJ5KntPeTr4115aXD Calendário Nacional de Vacinação - Adolescentes e jovens.pdf
Processing file 1S4ARdDZbj56QeLFfJN4cX_vW3Ul4Y5ht Calendário Nacional de Vacinação - Adulto.pdf
Processing file 1pjcS1pIapSgvGCbKnK3EA7IuGFU0KkMN Calendário Nacional de Vacinação - Criança.pdf
Processing file 1b0UW30XMGyYIP3YIfIzww84eclkC9hnT Calendário Nacional de Vacinação - Gestante.pdf
Processing file 1EMYvlJlbZwcbYOwV5CdGKBRkCypLd5nM Calendário Nacional de Vacinação - Idoso.pdf
Processing file 1ESYuyCsIrP24r9bXWtCpP5nyoX5OYreg Cartilha - Exames Preventivos do Câncer de Colo do Útero.pdf
Processing file 1dVdzMFpIryGneOKOoVnGikuqbOj-1Zhg Cartilha diabetes.pdf
Processing file 1flF4Gy5dpPSModcpLuk9F4vIi1NKPFVu Cartilha envelhecimento saudável.pdf
Processing file 1oaOwBNU

Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1PrmCOakvFOUrJeGxfRvSYIjvSxAyXogJ
To: /content/previa/docs/2197-UN-CARTILHA-CUIDADOS COM MEDICAMENTOS V3_compressed.pdf
100%|██████████| 927k/927k [00:00<00:00, 121MB/s]
Downloading...
From: https://drive.google.com/uc?id=1-Kk6XOBpeYoitB9Y5hsm_6igjTR_Yy5x
To: /content/previa/docs/caderno_tematico_pse_doencas_negligenciadas.pdf
100%|██████████| 6.29M/6.29M [00:00<00:00, 19.2MB/s]
Downloading...
From: https://drive.google.com/uc?id=1O9VjOYc1s8tsPlWzJ5KntPeTr4115aXD
To: /content/previa/docs/Calendário Nacional de Vacinação - Adolescentes e jovens.pdf
100%|██████████| 308k/308k [00:00<00:00, 81.7MB/s]
Downloading...
From: https://drive.google.com/uc?id=1S4ARdDZbj56QeLFfJN4cX_vW3Ul4Y5ht
To: /content/previa/docs/Calendário Nacional de Vacinação - Adulto.pdf
100%|██████████| 299k/299k [00:00<00:00, 82.6MB/s]
Downloading...
From: https://d


38 PDFs baixados:
  2197-UN-CARTILHA-CUIDADOS COM MEDICAMENTOS V3_compressed.pdf (905 KB)
  CARTILHA OBESIDADE VersaoD.pdf (427 KB)
  CARTILHA PERSONAL ODONTO - V4.pdf 1.pdf (2228 KB)
  Calendário Nacional de Vacinação - Adolescentes e jovens.pdf (301 KB)
  Calendário Nacional de Vacinação - Adulto.pdf (292 KB)
  Calendário Nacional de Vacinação - Criança.pdf (363 KB)
  Calendário Nacional de Vacinação - Gestante.pdf (294 KB)
  Calendário Nacional de Vacinação - Idoso.pdf (303 KB)
  Cartilha - Exames Preventivos do Câncer de Colo do Útero.pdf (2189 KB)
  Cartilha Hipertensão Arterial Sistêmica.pdf (2963 KB)
  Cartilha Programa de Tratamento ao Tabagismo UNP.pdf (4567 KB)
  Cartilha Saúde Bucal.pdf (738 KB)
  Cartilha diabetes.pdf (3162 KB)
  Cartilha envelhecimento saudável.pdf (4485 KB)
  Cartilha gestação saudável.pdf (2222 KB)
  Cartilha hábitos saudáveis.pdf (682 KB)
  Cartilha saúde da mulher.pdf (1017 KB)
  Cartilha saúde emocional.pdf (1340 KB)
  Cartilha-da-saude-da-mulher-2.p


Download completed


In [18]:
# Embedding português leve
print(' Carregando embedding...')
embeddings = HuggingFaceEmbeddings(
    model_name='neuralmind/bert-base-portuguese-cased',
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)
print('Embedding pronto!\n')

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300, chunk_overlap=50,
    separators=['\n\n', '\n', '. ', ' ', '']
)

def chunk_valido(texto, source_file, page):
    texto = texto.strip()
    if len(texto) < 80: return False
    letras = sum(c.isalpha() for c in texto)
    if letras < len(texto) * 0.4: return False
    if any(p in texto.lower()[:80] for p in ['referências', 'bibliography', 'miolo.indd']):
        return False
    return True

todos_chunks = []
docs_dir = '/content/previa/docs'
pdfs = sorted([f for f in os.listdir(docs_dir) if f.endswith('.pdf')])
print(f'Processando {len(pdfs)} PDFs...\n')

for pdf in pdfs:
    try:
        loader  = PyPDFLoader(f'{docs_dir}/{pdf}')
        paginas = loader.load()
        chunks  = splitter.split_documents(paginas)
        validos = []
        for chunk in chunks:
            chunk.metadata['source_file'] = pdf
            if chunk_valido(chunk.page_content, pdf, chunk.metadata.get('page', -1)):
                validos.append(chunk)
        todos_chunks.extend(validos)
        print(f'  {pdf}: {len(validos)} chunks')
    except Exception as e:
        print(f'  {pdf}: {e}')

print(f'\nTotal: {len(todos_chunks)} chunks')

 Carregando embedding...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/647 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Embedding pronto!

Processando 38 PDFs...

  2197-UN-CARTILHA-CUIDADOS COM MEDICAMENTOS V3_compressed.pdf: 80 chunks
  CARTILHA OBESIDADE VersaoD.pdf: 60 chunks
  CARTILHA PERSONAL ODONTO - V4.pdf 1.pdf: 55 chunks
  Calendário Nacional de Vacinação - Adolescentes e jovens.pdf: 13 chunks
  Calendário Nacional de Vacinação - Adulto.pdf: 8 chunks
  Calendário Nacional de Vacinação - Criança.pdf: 24 chunks
  Calendário Nacional de Vacinação - Gestante.pdf: 7 chunks
  Calendário Nacional de Vacinação - Idoso.pdf: 9 chunks
  Cartilha - Exames Preventivos do Câncer de Colo do Útero.pdf: 137 chunks
  Cartilha Hipertensão Arterial Sistêmica.pdf: 0 chunks
  Cartilha Programa de Tratamento ao Tabagismo UNP.pdf: 34 chunks
  Cartilha Saúde Bucal.pdf: 42 chunks
  Cartilha diabetes.pdf: 0 chunks
  Cartilha envelhecimento saudável.pdf: 35 chunks
  Cartilha gestação saudável.pdf: 62 chunks
  Cartilha hábitos saudáveis.pdf: 36 chunks
  Cartilha saúde da mulher.pdf: 40 chunks
  Cartilha saúde emocional.p

In [19]:
# Verificar PDFs com 0 chunks

pdfs_zero = [
    'Cartilha Hipertensão Arterial Sistêmica.pdf',
    'Cartilha diabetes.pdf',
]

for pdf in pdfs_zero:
    caminho = f'/content/previa/docs/{pdf}'
    tamanho = os.path.getsize(caminho) // 1024
    print(f'\n{pdf} ({tamanho} KB)')
    try:
        loader  = PyPDFLoader(caminho)
        paginas = loader.load()
        print(f'   Páginas: {len(paginas)}')
        for i, pag in enumerate(paginas[:3]):
            print(f'   Pág {i}: {len(pag.page_content)} chars — "{pag.page_content[:100]}"')
    except Exception as e:
        print(f'   Erro: {e}')


Cartilha Hipertensão Arterial Sistêmica.pdf (2963 KB)
   Páginas: 12
   Pág 0: 0 chars — ""
   Pág 1: 0 chars — ""
   Pág 2: 0 chars — ""

Cartilha diabetes.pdf (3162 KB)
   Páginas: 12
   Pág 0: 0 chars — ""
   Pág 1: 0 chars — ""
   Pág 2: 0 chars — ""


In [20]:
# OCR nos PDFs problemáticos, para extrair texto de imagens

def extrair_texto_ocr(caminho_pdf):
    """Extrai texto de PDF escaneado via OCR."""
    try:
        imagens = convert_from_path(caminho_pdf, dpi=200)
        texto_total = ''
        for img in imagens:
            texto = pytesseract.image_to_string(img, lang='por')
            texto_total += texto + '\n\n'
        return texto_total
    except Exception as e:
        return f'Erro OCR: {e}'

pdfs_zero = [
    'Cartilha Hipertensão Arterial Sistêmica.pdf',
    'Cartilha diabetes.pdf',
]

for pdf in pdfs_zero:
    caminho = f'/content/previa/docs/{pdf}'
    print(f'OCR em {pdf}...')
    texto = extrair_texto_ocr(caminho)
    print(f'   Extraídos {len(texto)} chars')
    print(f'   Amostra: {texto[:200]}')
    print()

OCR em Cartilha Hipertensão Arterial Sistêmica.pdf...
   Extraídos 4245 chars
   Amostra: VIVER BEM

HIPERTENSÃO
ABR os

 


[=]: N [=] Leia o código e
A assista a história

de seu Juvenal e

dona Conceição. | |

 


Seu Juvenal e dona Conceição são casados há 40
anos. E há 40 anos, toda

OCR em Cartilha diabetes.pdf...
   Extraídos 6181 chars
   Amostra: VIVER BEM

E DIABETES IE

 


oo

Leia o código e
assista a história
de Luiza Antônia:

EE

     
 

 


Se você olhar para Luiza Antônia, vai ver que ela é
uma mulher como todas as outras. E nem va



In [21]:
# Indexar FAISS em lotes

faiss_dir = '/content/previa/faiss_db'
shutil.rmtree(faiss_dir, ignore_errors=True)
os.makedirs(faiss_dir, exist_ok=True)

LOTE = 200
vectorstore = None
print(f' Indexando {len(todos_chunks)} chunks em lotes de {LOTE}...\n')

for i in range(0, len(todos_chunks), LOTE):
    lote = todos_chunks[i:i+LOTE]
    if vectorstore is None:
        vectorstore = FAISS.from_documents(lote, embeddings)
    else:
        vectorstore.add_documents(lote)
    print(f'  {min(i+LOTE, len(todos_chunks))}/{len(todos_chunks)}')

vectorstore.save_local(faiss_dir)
print('\nIndexação concluída!')

 Indexando 8699 chunks em lotes de 200...

  200/8699
  400/8699
  600/8699
  800/8699
  1000/8699
  1200/8699
  1400/8699
  1600/8699
  1800/8699
  2000/8699
  2200/8699
  2400/8699
  2600/8699
  2800/8699
  3000/8699
  3200/8699
  3400/8699
  3600/8699
  3800/8699
  4000/8699
  4200/8699
  4400/8699
  4600/8699
  4800/8699
  5000/8699
  5200/8699
  5400/8699
  5600/8699
  5800/8699
  6000/8699
  6200/8699
  6400/8699
  6600/8699
  6800/8699
  7000/8699
  7200/8699
  7400/8699
  7600/8699
  7800/8699
  8000/8699
  8200/8699
  8400/8699
  8600/8699
  8699/8699

Indexação concluída!


In [22]:
#  Adicionar chunks OCR ao índice

splitter_ocr = RecursiveCharacterTextSplitter(
    chunk_size=300, chunk_overlap=50
)

chunks_ocr = []
for pdf in pdfs_zero:
    caminho = f'/content/previa/docs/{pdf}'
    texto   = extrair_texto_ocr(caminho)
    if len(texto) < 100:
        print(f'{pdf}: texto insuficiente')
        continue

    # Divide em chunks
    doc    = Document(page_content=texto, metadata={'source_file': pdf, 'page': 0, 'ocr': True})
    chunks = splitter_ocr.split_documents([doc])
    for chunk in chunks:
        chunk.metadata['source_file'] = pdf

    # Filtra chunks ruins
    validos = [c for c in chunks if len(c.page_content.strip()) >= 80]
    chunks_ocr.extend(validos)
    print(f'{pdf}: {len(validos)} chunks via OCR')

if chunks_ocr:
    vectorstore.add_documents(chunks_ocr)
    vectorstore.save_local('/content/previa/faiss_db')
    print(f'\n{len(chunks_ocr)} chunks adicionados!')
    print(f'   Total no índice agora: {vectorstore.index.ntotal} chunks')
else:
    print('Nenhum chunk gerado')

Cartilha Hipertensão Arterial Sistêmica.pdf: 18 chunks via OCR
Cartilha diabetes.pdf: 27 chunks via OCR

45 chunks adicionados!
   Total no índice agora: 8744 chunks


In [23]:
# Diagnóstico do corpus

docs_dir = '/content/previa/docs'
pdfs = [f for f in os.listdir(docs_dir) if f.endswith('.pdf')]

print(f'{len(pdfs)} PDFs no corpus:\n')
for pdf in sorted(pdfs):
    tamanho = os.path.getsize(f'{docs_dir}/{pdf}') // 1024
    print(f'  {pdf} ({tamanho} KB)')

print(f'\nTotal de chunks indexados: {len(todos_chunks)}')

# Mostra distribuição por documento
fontes = Counter(c.metadata.get('source_file', '?') for c in todos_chunks)
print('\nChunks por documento:')
for fonte, qtd in fontes.most_common():
    print(f'  {fonte}: {qtd} chunks')

38 PDFs no corpus:

  2197-UN-CARTILHA-CUIDADOS COM MEDICAMENTOS V3_compressed.pdf (905 KB)
  CARTILHA OBESIDADE VersaoD.pdf (427 KB)
  CARTILHA PERSONAL ODONTO - V4.pdf 1.pdf (2228 KB)
  Calendário Nacional de Vacinação - Adolescentes e jovens.pdf (301 KB)
  Calendário Nacional de Vacinação - Adulto.pdf (292 KB)
  Calendário Nacional de Vacinação - Criança.pdf (363 KB)
  Calendário Nacional de Vacinação - Gestante.pdf (294 KB)
  Calendário Nacional de Vacinação - Idoso.pdf (303 KB)
  Cartilha - Exames Preventivos do Câncer de Colo do Útero.pdf (2189 KB)
  Cartilha Hipertensão Arterial Sistêmica.pdf (2963 KB)
  Cartilha Programa de Tratamento ao Tabagismo UNP.pdf (4567 KB)
  Cartilha Saúde Bucal.pdf (738 KB)
  Cartilha diabetes.pdf (3162 KB)
  Cartilha envelhecimento saudável.pdf (4485 KB)
  Cartilha gestação saudável.pdf (2222 KB)
  Cartilha hábitos saudáveis.pdf (682 KB)
  Cartilha saúde da mulher.pdf (1017 KB)
  Cartilha saúde emocional.pdf (1340 KB)
  Cartilha-da-saude-da-mulher-2.

In [24]:
# Validar qualidade com perguntas

perguntas = [
    'Quais exames preventivos para hipertensão arterial?',
    'Mamografia a partir de qual idade?',
    'Rastreamento câncer do colo do útero periodicidade',
    'Recomendações atividade física adultos minutos por semana',
    'Exames de sangue recomendados prevenção diabetes',
]

print('Validação de qualidade do RAG:\n')
for pergunta in perguntas:
    resultados = vectorstore.similarity_search(pergunta, k=1)
    doc = resultados[0]
    fonte = doc.metadata.get('source_file', '?')
    pagina = doc.metadata.get('page', '?')
    trecho = doc.page_content[:200].replace('\n', ' ')
    print(f'{pergunta}')
    print(f'   {fonte} | pág. {pagina}')
    print(f'   {trecho}')
    print()

Validação de qualidade do RAG:

Quais exames preventivos para hipertensão arterial?
   hipertensao_cab37.pdf | pág. 47
   o paciente deve realizar, via encaminhamento, investigação de hiperaldosteronismo primário.  • A dosagem do colesterol e da glicemia visa detectar outros fatores que potencializam o  risco cardiovascu

Mamografia a partir de qual idade?
   rastreamento_cap29.pdf | pág. 72
   de mama – recomendações do INCA População-alvo Periodicidade dos exames de rastreamento Mulheres de 40 a 49 anos ECM anual e, se este estiver alterado, mamografia. Mulheres de 50 a 69 anos ECM anual e

Rastreamento câncer do colo do útero periodicidade
   rastreamento_cancer_colo_utero.pdf | pág. 29
   29 Capítulo 1 - rastreio de lesões precursoras do  câncer do colo do útero Método, cobertura, população-alvo e periodicidade  Decisões de quem rastrear e de quando rastrear para detecção das lesões pr

Recomendações atividade física adultos minutos por semana
   hipertensao_cab37.pdf | pág. 101
  

## Corpus Indexado

### Rastreamento e câncer

| Documento | Fonte |
|-----------|-------|
| Diretrizes Brasileiras – Rastreamento do Câncer do Colo do Útero | INCA / Ministério da Saúde |
| Parâmetros Técnicos – Rastreamento do Câncer de Mama | INCA / Ministério da Saúde |
| Cartilha – Exames Preventivos do Câncer de Colo do Útero | Ministério da Saúde |
| Cartilha Câncer de Mama 2025 | INCA / Ministério da Saúde |
| Controle dos Cânceres do Colo do Útero e de Mama | INCA |
| Plano de Ação – Redução do Câncer do Colo do Útero | Ministério da Saúde |
| Situação do Câncer no Brasil | INCA / Ministério da Saúde |
| Boletim Temático – Prevenção do Câncer do Colo do Útero (2023) | Ministério da Saúde |
| Próstata – Prevenção e Rastreamento | Ministério da Saúde |
| Exames Preventivos | Ministério da Saúde |

---

### Doenças crônicas e atenção básica

| Documento | Fonte |
|-----------|-------|
| Hipertensão Arterial Sistêmica – CAB 37 | Ministério da Saúde |
| Cartilha Hipertensão Arterial Sistêmica | Ministério da Saúde |
| Cartilha – Cuidados com Hipertensão em Idosos | Ministério da Saúde |
| Cartilha Diabetes | Ministério da Saúde |
| Estratégias para o Cuidado da Pessoa com Doença Crônica – CAB 35 | Ministério da Saúde |
| Rastreamento na Atenção Primária – CAP 29 | Ministério da Saúde |
| Promoção da Saúde e Prevenção de Riscos e Doenças | ANS / Ministério da Saúde |
| Caderno Temático – Doenças Negligenciadas | Ministério da Saúde |
| Cartilha Obesidade | Ministério da Saúde |
| Cartilha – Cuidados com Medicamentos | Ministério da Saúde |

---

### Saúde da mulher e gestação

| Documento | Fonte |
|-----------|-------|
| Cartilha Saúde da Mulher | Ministério da Saúde |
| Cartilha Saúde da Mulher 2 | Ministério da Saúde |
| Cartilha Saúde da Mulher 3 | Ministério da Saúde |
| Cartilha Gestação Saudável | Ministério da Saúde |

---

### Vacinação

| Documento | Fonte |
|-----------|-------|
| Calendário Nacional de Vacinação – Adulto | Ministério da Saúde |
| Calendário Nacional de Vacinação – Idoso | Ministério da Saúde |
| Calendário Nacional de Vacinação – Gestante | Ministério da Saúde |
| Calendário Nacional de Vacinação – Adolescentes e Jovens | Ministério da Saúde |
| Calendário Nacional de Vacinação – Criança | Ministério da Saúde |

---

### Nutrição e estilo de vida

| Documento | Fonte |
|-----------|-------|
| Guia Alimentar para a População Brasileira – 2ª ed. | Ministério da Saúde |
| OMS – Alimentação Saudável | Organização Mundial da Saúde |
| OMS – Diretrizes de Atividade Física | Organização Mundial da Saúde |
| Cartilha Hábitos Saudáveis | Ministério da Saúde |
| Cartilha Envelhecimento Saudável | Ministério da Saúde |

---

### Saúde bucal, emocional e outros

| Documento | Fonte |
|-----------|-------|
| Cartilha Saúde Bucal | Ministério da Saúde |
| Cartilha Saúde Emocional | Ministério da Saúde |
| Cartilha Programa de Tratamento ao Tabagismo | UNP / Ministério da Saúde |
| Cartilha Personal Odonto | Ministério da Saúde |

---

## Estatísticas do Corpus

- **Total de documentos indexados:** 39
- **Total de chunks válidos:** ~ 9000

**Distribuição por categoria:**

| Categoria | Documentos |
|-----------|-----------|
| Rastreamento e câncer | 10 |
| Doenças crônicas e atenção básica | 10 |
| Saúde da mulher e gestação | 4 |
| Vacinação | 5 |
| Nutrição e estilo de vida | 5 |
| Saúde bucal, emocional e outros | 4 |
| **Total** | **39** |

## Agentes

In [25]:
# Iniciar Ollama

subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

try:
    requests.get('http://localhost:11434')
    print('Ollama rodando')
except:
    print('Ollama não iniciou — tente novamente')

Ollama rodando


In [26]:
# Carregar LLM e VectorStore

# LLM
llm = ChatOllama(model='qwen2.5:7b', temperature=0.2)
print('LLM carregado')

# Embeddings
embeddings = HuggingFaceEmbeddings(
    model_name='neuralmind/bert-base-portuguese-cased',
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)

# VectorStore
vectorstore = FAISS.load_local(
    '/content/previa/faiss_db',
    embeddings,
    allow_dangerous_deserialization=True
)
retriever = vectorstore.as_retriever(search_kwargs={'k': 8})
print('VectorStore carregado')

LLM carregado


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


VectorStore carregado


In [27]:
# Definir o Estado do Grafo
# O State é o objeto que circula entre todos os agentes
# Cada agente lê e escreve campos nesse dicionário

class PrevIAState(TypedDict):
    # Entrada do usuário
    pergunta: str
    perfil: Optional[dict]          # idade, sexo, histórico familiar etc.

    # Rota decidida pelo Supervisor
    rota: str                        # 'qa' ou 'automacao'

    # Documentos recuperados pelo Retriever
    documentos: List[Document]

    # Resposta gerada pelo Writer
    resposta: str

    # Resultado do Self-Check
    check_ok: bool
    tentativas: int                  # conta re-buscas do self-check

    # Saída final
    resposta_final: str

print('Estado do grafo definido!')

Estado do grafo definido!


In [28]:
# Agente Supervisor
# Decide se a pergunta é Q&A simples ou pedido de plano preventivo

def supervisor(state: PrevIAState) -> PrevIAState:
    pergunta = state['pergunta'].lower()

    # Palavras-chave que indicam pedido de plano/automação
    gatilhos_automacao = [
        'plano', 'planejamento', 'cronograma', 'calendário',
        'gerar plano', 'criar plano', 'meu perfil',
        'tenho histórico', 'preventivo anual'
    ]

    rota = 'automacao' if any(g in pergunta for g in gatilhos_automacao) else 'qa'

    print(f'🧭 Supervisor → rota: {rota}')
    return {**state, 'rota': rota, 'tentativas': 0}

print('Supervisor definido!')

Supervisor definido!


In [29]:
# Agente Retriever
# Busca os trechos mais relevantes no FAISS

def retriever_agent(state: PrevIAState) -> PrevIAState:
    pergunta = state['pergunta']

    # Se tiver perfil, enriquece a query
    if state.get('perfil'):
        perfil = state['perfil']
        query_enriquecida = (
            f"{pergunta} "
            f"idade {perfil.get('idade', '')} "
            f"sexo {perfil.get('sexo', '')} "
            f"fatores de risco {perfil.get('historico', '')}"
        )
    else:
        query_enriquecida = pergunta

    docs = retriever.invoke(query_enriquecida)
    print(f'📚 Retriever → {len(docs)} documentos recuperados')
    for d in docs:
        print(f'   • {d.metadata.get("source_file")} pág.{d.metadata.get("page")}')

    return {**state, 'documentos': docs}

print('Retriever definido')

Retriever definido


In [30]:
# Agente Writer (com citações)
# Gera a resposta formatada com citações das fontes

PROMPT_WRITER = ChatPromptTemplate.from_template("""
Você é o PrevIA, assistente de medicina preventiva brasileiro.

PERGUNTA: {pergunta}

TRECHOS DOS DOCUMENTOS:
{contexto}

INSTRUÇÕES OBRIGATÓRIAS:
1. Leia todos os trechos com atenção
2. Extraia QUALQUER informação relevante para a pergunta, mesmo que parcial
3. Se encontrar idade, faixa etária, frequência ou periodicidade — inclua na resposta
4. Responda em português com marcadores (•)
5. Seja direto: comece a resposta com a informação principal
6. Cite as fontes no final: [Fonte: arquivo, pág. X]
7. Só diga "não encontrei" se os trechos realmente não tiverem NADA relacionado

RESPOSTA:""")

def writer_agent(state: PrevIAState) -> PrevIAState:
    docs = state['documentos']

    # Formata o contexto com metadados para citação
    contexto = ''
    for i, doc in enumerate(docs):
        fonte = doc.metadata.get('source_file', 'desconhecido')
        pagina = doc.metadata.get('page', '?')
        contexto += f'[Trecho {i+1} — {fonte}, pág. {pagina}]\n'
        contexto += doc.page_content + '\n\n'

    chain = PROMPT_WRITER | llm
    resultado = chain.invoke({
        'pergunta': state['pergunta'],
        'contexto': contexto
    })

    print('✍️  Writer → resposta gerada')
    return {**state, 'resposta': resultado.content}

print('Writer definido')

Writer definido


In [31]:
# Agente Self-Check (anti-alucinação)
# Verifica se a resposta é suportada pelos documentos recuperados
# Se não for, tenta uma nova busca (máx. 1 vez) ou recusa

PROMPT_SELFCHECK = ChatPromptTemplate.from_template("""
Você é um verificador de evidências médicas.

Analise se a RESPOSTA usa informações que estão presentes nos TRECHOS.

TRECHOS:
{contexto}

RESPOSTA A VERIFICAR:
{resposta}

Critério de aprovação:
- APROVADO se a resposta menciona pelo menos 1 informação presente nos trechos
- APROVADO se a resposta diz que não encontrou evidência (é uma resposta honesta)
- REPROVADO apenas se a resposta inventar informações que CONTRADIZEM os trechos

Responda APENAS com: APROVADO ou REPROVADO
Resultado:""")

def self_check(state: PrevIAState) -> PrevIAState:
    docs = state['documentos']

    # Se não há documentos, aprova direto
    if not docs:
        return {**state, 'check_ok': True, 'tentativas': state.get('tentativas', 0) + 1}

    contexto = '\n\n'.join([d.page_content[:300] for d in docs])  # limita tamanho

    chain = PROMPT_SELFCHECK | llm
    resultado = chain.invoke({
        'contexto': contexto,
        'resposta': state['resposta'][:500]  # limita tamanho da resposta
    })

    veredicto = resultado.content.strip().upper()
    check_ok  = 'REPROVADO' not in veredicto  # benefício da dúvida — aprova se não for explicitamente reprovado
    tentativas = state.get('tentativas', 0)

    print(f'🔍 Self-Check → {"✅ APROVADO" if check_ok else "❌ REPROVADO"} (tentativa {tentativas+1})')
    return {**state, 'check_ok': check_ok, 'tentativas': tentativas + 1}

def rota_apos_check(state: PrevIAState) -> str:
    """Decide o que fazer após o self-check."""
    if state['check_ok']:
        return 'aprovado'
    elif state['tentativas'] < 2:
        print('Self-Check → re-buscando evidências...')
        return 're_busca'
    else:
        print('Self-Check → recusando resposta por falta de evidências')
        return 'recusar'

print(' Self-Check definido')

 Self-Check definido


In [32]:
# Agente Safety
# Adiciona disclaimer médico obrigatório e filtra conteúdo perigoso

DISCLAIMER = """
---
⚠️ **Aviso importante:** Este sistema fornece informações educativas baseadas em
diretrizes públicas do Ministério da Saúde e OMS. **Não substitui consulta com
profissional de saúde.** Converse sempre com seu médico antes de tomar decisões
sobre exames ou tratamentos.
"""

TERMOS_PERIGOSOS = [
    'prescrever', 'tomar remédio', 'dose de', 'medicamento para',
    'diagnóstico de', 'você tem', 'você está com'
]

def safety_agent(state: PrevIAState) -> PrevIAState:
    resposta = state.get('resposta', '')

    # Verifica se a resposta tenta fazer diagnóstico/prescrição
    resposta_lower = resposta.lower()
    if any(termo in resposta_lower for termo in TERMOS_PERIGOSOS):
        resposta = (
            'Não posso fornecer diagnósticos ou prescrições médicas. '
            'Por favor, consulte um profissional de saúde para orientações específicas.\n\n'
            + resposta
        )

    # Adiciona disclaimer
    resposta_final = resposta + DISCLAIMER
    print('🛡️  Safety → disclaimer adicionado')
    return {**state, 'resposta_final': resposta_final}

def recusar(state: PrevIAState) -> PrevIAState:
    """Resposta de recusa quando o self-check falha duas vezes."""
    recusa = (
        'Não encontrei evidência suficiente nas fontes indexadas para '
        'responder com segurança a esta pergunta. '
        'Recomendo consultar diretamente as diretrizes do Ministério da Saúde '
        'ou um profissional de saúde.'
    )
    return {**state, 'resposta': recusa, 'resposta_final': recusa + DISCLAIMER}

print('Safety definido')

Safety definido


In [33]:
def selecionar_calendarios(perfil: dict) -> list:
    idade     = perfil.get('idade', 30)
    historico = perfil.get('historico', '').lower()
    condicoes = perfil.get('condicoes', '').lower()

    gestante  = any(p in historico + condicoes for p in ['gestante', 'grávida', 'gravida', 'gravidez', 'gestação'])

    calendarios = []

    if idade < 12:
        calendarios.append('Calendário Nacional de Vacinação - Criança.pdf')
    elif idade < 20:
        calendarios.append('Calendário Nacional de Vacinação - Adolescentes e jovens.pdf')
    elif idade >= 60:
        calendarios.append('Calendário Nacional de Vacinação - Idoso.pdf')
    else:
        calendarios.append('Calendário Nacional de Vacinação - Adulto.pdf')

    if gestante and 'Gestante' not in calendarios[0]:
        calendarios.append('Calendário Nacional de Vacinação - Gestante.pdf')

    return calendarios


In [34]:
# Automation Agent (gera plano preventivo)

def automation_agent(state: PrevIAState) -> PrevIAState:
    perfil      = state.get('perfil', {})
    calendarios = selecionar_calendarios(perfil)
    historico   = perfil.get('historico', 'nenhum')
    sexo        = perfil.get('sexo', '')
    idade       = perfil.get('idade', '')

    print(f'📅 Calendário selecionado: {calendarios}')

    queries = [
        f"exames preventivos {sexo} {idade} anos rastreamento",
        f"hábitos saudáveis prevenção doenças atividade física alimentação",
        f"exames monitoramento {historico}",
    ] + calendarios

    docs_vistos = set()
    todos_docs  = []
    for query in queries:
        for doc in retriever.invoke(str(query).strip()):
            chave = f"{doc.metadata.get('source_file')}_{doc.metadata.get('page')}"
            if chave not in docs_vistos:
                docs_vistos.add(chave)
                todos_docs.append(doc)
        if len(todos_docs) >= 15:
            break

    contexto = ''
    for doc in todos_docs:
        src = doc.metadata.get('source_file', '?')
        pag = doc.metadata.get('page', '?')
        contexto += f'[{src}, pág.{pag}]\n{doc.page_content}\n\n'

    perfil_str = '\n'.join([f'- {k}: {v}' for k, v in perfil.items()])
    cal_str    = ' e '.join(calendarios)

    PROMPT_AUTOMATION = ChatPromptTemplate.from_template("""
Você é o PrevIA, assistente de medicina preventiva brasileiro.

Gere um Plano Preventivo Anual completo para este perfil:
{perfil}

ATENÇÃO ESPECIAL: O perfil indica histórico de {historico}.
Inclua OBRIGATORIAMENTE na seção de Exames Recomendados:
- Exames específicos para monitoramento de {historico}
- Exemplo: se histórico de hipertensão → aferição de pressão arterial, perfil lipídico, função renal
- Exemplo: se histórico de diabetes → glicemia em jejum, hemoglobina glicada
- Exemplo: se sedentária → avaliação cardiovascular antes de iniciar exercícios

TRECHOS DOS DOCUMENTOS:
{contexto}

INSTRUÇÕES:
- Extraia dos trechos: exames com periodicidade, vacinas recomendadas
- Para VACINAS: use os trechos do calendário {cal_str}
- Para HÁBITOS PREVENTIVOS: liste comportamentos saudáveis que PREVINEM doenças,
  priorizando os relacionados a {historico}
- No cronograma: distribua os exames por trimestre com meses específicos
- Nunca escreva "ver médico para orientação" como item do cronograma

## Plano Preventivo Anual Personalizado

### Exames Recomendados
[exames gerais + exames específicos para {historico}]

### Vacinas
[vacinas do calendário {cal_str} encontradas nos trechos]

### Hábitos Preventivos
[hábitos extraídos dos trechos, priorizando os relacionados a {historico}]

### Cronograma Sugerido
[1 Trimestre Jan-Mar: atividades do periodo]
[2 Trimestre Abr-Jun: atividades do periodo]
[3 Trimestre Jul-Set: atividades do periodo]
[4 Trimestre Out-Dez: atividades do periodo]

### Fontes
[nome_arquivo.pdf, pag. X]
""")

    result = (PROMPT_AUTOMATION | llm).invoke({
        'perfil':    perfil_str,
        'contexto':  contexto,
        'cal_str':   cal_str,
        'historico': historico,
    })

    print('🤖 Automation Agent → plano gerado')
    return {**state, 'documentos': todos_docs, 'resposta': result.content, 'check_ok': True}

print('Automation Agent pronto')

Automation Agent pronto


In [35]:
# Montar o Grafo LangGraph

def rota_supervisor(state: PrevIAState) -> str:
    return state['rota']  # 'qa' ou 'automacao'

# Cria o grafo
grafo = StateGraph(PrevIAState)

# Adiciona nós
grafo.add_node('supervisor',       supervisor)
grafo.add_node('retriever',        retriever_agent)
grafo.add_node('writer',           writer_agent)
grafo.add_node('self_check',       self_check)
grafo.add_node('safety',           safety_agent)
grafo.add_node('recusar',          recusar)
grafo.add_node('automation_agent', automation_agent)

# Ponto de entrada
grafo.set_entry_point('supervisor')

# Supervisor decide a rota
grafo.add_conditional_edges(
    'supervisor',
    rota_supervisor,
    {
        'qa':        'retriever',
        'automacao': 'automation_agent'
    }
)

# Rota Q&A
grafo.add_edge('retriever', 'writer')
grafo.add_edge('writer',    'self_check')

# Self-check decide próximo passo
grafo.add_conditional_edges(
    'self_check',
    rota_apos_check,
    {
        'aprovado': 'safety',
        're_busca': 'retriever',  # tenta buscar de novo
        'recusar':  'recusar'
    }
)

# Rota Automação
grafo.add_edge('automation_agent', 'safety')

# Fim
grafo.add_edge('safety',  END)
grafo.add_edge('recusar', END)

# Compila
app = grafo.compile()
print('Grafo LangGraph compilado')

Grafo LangGraph compilado


In [36]:
# Função auxiliar para rodar o grafo

def perguntar(pergunta: str, perfil: dict = None):
    """Interface simples para interagir com o PrevIA."""
    print('=' * 60)
    print(f'Pergunta: {pergunta}')
    if perfil:
        print(f'Perfil: {perfil}')
    print('=' * 60)

    estado_inicial = {
        'pergunta':      pergunta,
        'perfil':        perfil,
        'rota':          '',
        'documentos':    [],
        'resposta':      '',
        'check_ok':      False,
        'tentativas':    0,
        'resposta_final': ''
    }

    resultado = app.invoke(estado_inicial)

    print('\n' + '=' * 60)
    print('RESPOSTA FINAL:')
    print('=' * 60)
    print(resultado['resposta_final'])
    return resultado

print('Função auxiliar pronta')

Função auxiliar pronta


In [ ]:
# Teste 1: Q&A simples

resultado = perguntar('A partir de qual idade devo fazer mamografia?')

Pergunta: A partir de qual idade devo fazer mamografia?
🧭 Supervisor → rota: qa
📚 Retriever → 8 documentos recuperados
   • Cartilha_saude_mulher.pdf pág.24
   • Cartilha_Cancer_Mama_2025_web_161025.pdf pág.14
   • Cartilha_Cancer_Mama_2025_web_161025.pdf pág.4
   • rastreamento_cap29.pdf pág.72
   • Cartilha_Cancer_Mama_2025_web_161025.pdf pág.12
   • Cartilha_Cancer_Mama_2025_web_161025.pdf pág.12
   • rastreamento_cap29.pdf pág.71
   • Cartilha-da-saude-da-mulher-2.pdf pág.5


In [ ]:
# Teste 2: Q&A com pergunta de rastreamento

resultado = perguntar('Com que frequência devo fazer o exame preventivo do colo do útero?')

In [ ]:
# Teste 3: Automação — Gerar Plano Preventivo
# rota de AUTOMAÇÃO

perfil_usuario = {
    'idade':    35,
    'sexo':     'feminino',
    'historico': 'hipertensão na família, sedentária',
    'condicoes': 'saudável'
}

resultado = perguntar(
    'Gere meu plano preventivo anual personalizado',
    perfil=perfil_usuario
)

### 📋 Resumo do grafo
| Agente | Função |
|--------|--------|
| Supervisor | Detecta rota: Q&A ou Automação |
| Retriever | Busca FAISS com query enriquecida |
| Writer | Gera resposta com citações |
| Self-Check | Valida se resposta tem suporte |
| Safety | Adiciona disclaimer médico |
| Automation | Gera plano preventivo personalizado |

## MCP

In [ ]:

embeddings = HuggingFaceEmbeddings(
    model_name='neuralmind/bert-base-portuguese-cased',
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)
vectorstore = FAISS.load_local(
    '/content/previa/faiss_db', embeddings,
    allow_dangerous_deserialization=True
)
retriever = vectorstore.as_retriever(search_kwargs={'k': 8})
llm = ChatOllama(model='qwen2.5:7b', temperature=0.1)

print('RAG e LLM carregados!')
print(f'   Embedding: neuralmind/bert-base-portuguese-cased')
print(f'   Chunks: {vectorstore.index.ntotal} | TOP_K: 8')

In [ ]:
logger = logging.getLogger('mcp_audit')
logger.setLevel(logging.INFO)
logger.handlers.clear()

fh = logging.FileHandler('/content/previa/mcp_audit.log')
fh.setFormatter(logging.Formatter('%(asctime)s | %(message)s'))
logger.addHandler(fh)

print('Logger de auditoria configurado!')

In [ ]:
os.makedirs('/content/previa/src/mcp', exist_ok=True)

server_code = (
    "# health_checklist_server.py\n"
    "# Servidor MCP do PrevIA\n"
    "# Seguranca: allowlist, sem acesso a disco/internet, logs de auditoria\n\n"
    "import asyncio, json, logging\n"
    "from mcp.server import Server\n"
    "from mcp.server.stdio import stdio_server\n"
    "from mcp import types\n\n"
    "logging.basicConfig(filename='/content/previa/mcp_audit.log',\n"
    "    level=logging.INFO, format='%(asctime)s | %(message)s')\n\n"
    "ALLOWED_TOOLS = {'generate_preventive_checklist'}\n"
    "server = Server('health-checklist')\n\n"
    "@server.list_tools()\n"
    "async def list_tools():\n"
    "    return [types.Tool(\n"
    "        name='generate_preventive_checklist',\n"
    "        description='Gera checklist preventivo baseado em diretrizes do MS e OMS.',\n"
    "        inputSchema={'type':'object','properties':{\n"
    "            'age':{'type':'integer'},'sex':{'type':'string'},\n"
    "            'risk_factors':{'type':'string'}},'required':['age','sex']})]\n\n"
    "@server.call_tool()\n"
    "async def call_tool(name, arguments):\n"
    "    if name not in ALLOWED_TOOLS:\n"
    "        logging.warning(f'BLOQUEADO | tool={name}')\n"
    "        return [types.TextContent(type='text', text=f'Tool {name} nao permitida.')]\n"
    "    logging.info(f'CHAMADA | tool={name} | args={json.dumps(arguments)}')\n"
    "    resultado = await gerar_checklist(**arguments)\n"
    "    logging.info(f'SUCESSO | tool={name}')\n"
    "    return [types.TextContent(type='text', text=resultado)]\n\n"
    "async def gerar_checklist(age, sex, risk_factors='nenhum'):\n"
    "    return f'Checklist para {age} anos, {sex}, riscos: {risk_factors}'\n\n"
    "async def main():\n"
    "    async with stdio_server() as (r, w):\n"
    "        await server.run(r, w, server.create_initialization_options())\n\n"
    "if __name__ == '__main__':\n"
    "    asyncio.run(main())\n"
)

with open('/content/previa/src/mcp/health_checklist_server.py', 'w') as f:
    f.write(server_code)

print('Servidor MCP salvo em /content/previa/src/mcp/health_checklist_server.py')

In [ ]:

ALLOWED_TOOLS = {'generate_preventive_checklist'}

@tool
def generate_preventive_checklist(age: int, sex: str, risk_factors: str = 'nenhum') -> str:
    """
    [MCP: health-checklist] Gera checklist preventivo personalizado.
    Args:
        age: idade do paciente
        sex: sexo (masculino/feminino)
        risk_factors: fatores de risco separados por vírgula
    """
    logger.info(f'CHAMADA | tool=generate_preventive_checklist | age={age} sex={sex} risk={risk_factors}')

    docs = retriever.invoke(f'exames preventivos {sex} {age} anos {risk_factors}')
    contexto, fontes = '', []
    for doc in docs:
        src = doc.metadata.get('source_file', '?')
        pag = doc.metadata.get('page', '?')
        contexto += f'[{src}, pag.{pag}]\n{doc.page_content}\n\n'
        fontes.append(f'{src} pag.{pag}')

    prompt = ChatPromptTemplate.from_template("""
Gere checklist preventivo para:
- Idade: {age} anos | Sexo: {sex} | Fatores de risco: {risk_factors}

Use APENAS os trechos. Nao invente exames ou vacinas.
Se nao houver informacao: Consulte um profissional de saude.

TRECHOS:
{contexto}

EXAMES: [apenas os que aparecem nos trechos]
VACINAS: [apenas as que aparecem nos trechos]
HABITOS: [apenas os que aparecem nos trechos]
FONTES: {fontes}
""")

    result = (prompt | llm).invoke({
        'age': age, 'sex': sex, 'risk_factors': risk_factors,
        'contexto': contexto, 'fontes': ', '.join(fontes)
    })

    logger.info(f'SUCESSO | tool=generate_preventive_checklist | chars={len(result.content)}')
    return result.content

print(f'Tool MCP registrada')
print(f'   Allowlist: {ALLOWED_TOOLS}')

In [ ]:
print('Testando tool MCP...\n')

resultado = generate_preventive_checklist.invoke({
    'age':          45,
    'sex':          'feminino',
    'risk_factors': 'historico familiar de cancer de mama, hipertensao'
})

print('Checklist gerado:')
print(resultado)

print('\n' + '='*50)
print('Log de auditoria:')
print('='*50)
with open('/content/previa/mcp_audit.log', 'r') as f:
    print(f.read())

In [ ]:
# Compilar grafo com MCP integrado

class PrevIAState(TypedDict):
    pergunta:       str
    perfil:         Optional[dict]
    rota:           str
    documentos:     List[Document]
    resposta:       str
    check_ok:       bool
    tentativas:     int
    resposta_final: str

def rota_supervisor(state): return state['rota']

grafo = StateGraph(PrevIAState)
grafo.add_node('supervisor',       supervisor)
grafo.add_node('retriever',        retriever_agent)
grafo.add_node('writer',           writer_agent)
grafo.add_node('self_check',       self_check)
grafo.add_node('safety',           safety_agent)
grafo.add_node('recusar',          recusar)
grafo.add_node('automation_agent', automation_agent)  # versão com MCP e histórico

grafo.set_entry_point('supervisor')
grafo.add_conditional_edges('supervisor', rota_supervisor,
    {'qa': 'retriever', 'automacao': 'automation_agent'})
grafo.add_edge('retriever',        'writer')
grafo.add_edge('writer',           'self_check')
grafo.add_conditional_edges('self_check', rota_apos_check,
    {'aprovado': 'safety', 're_busca': 'retriever', 'recusar': 'recusar'})
grafo.add_edge('automation_agent', 'safety')
grafo.add_edge('safety',           END)
grafo.add_edge('recusar',          END)

app = grafo.compile()
print('Grafo com MCP compilado!')

In [ ]:
# Teste completo do fluxo de automação via MCP

perfil = {
    'idade':     45,
    'sexo':      'feminino',
    'historico': 'historico familiar de cancer de mama, hipertensao',
    'condicoes': 'saudavel'
}

estado = {
    'pergunta':       'Gere meu plano preventivo anual personalizado',
    'perfil':         perfil,
    'rota':           '',
    'documentos':     [],
    'resposta':       '',
    'check_ok':       False,
    'tentativas':     0,
    'resposta_final': ''
}

resultado = app.invoke(estado)
print('\n' + '='*60)
print('RESPOSTA FINAL (via MCP):')
print('='*60)
print(resultado['resposta_final'])

## MCP integrado!

| Item | Detalhe |
|------|---------|
| Servidor | `health-checklist` (MCP próprio — Opção 1 do enunciado) |
| Tool exposta | `generate_preventive_checklist` |
| Allowlist | Apenas 1 tool permitida |
| Logs | Toda chamada registrada em `mcp_audit.log` |
| Acesso negado | Disco, internet, execução de código |

## Avaliação

In [ ]:
# Carregar RAG com embedding português

embeddings = HuggingFaceEmbeddings(
    model_name='neuralmind/bert-base-portuguese-cased',
    model_kwargs={'device': 'cpu'},
    encode_kwargs={'normalize_embeddings': True}
)
vectorstore = FAISS.load_local(
    '/content/previa/faiss_db', embeddings,
    allow_dangerous_deserialization=True
)
retriever = vectorstore.as_retriever(search_kwargs={'k': 8})
llm = ChatOllama(model='qwen2.5:7b', temperature=0.2)

print(f'RAG carregado!')
print(f'   Embedding: neuralmind/bert-base-portuguese-cased')
print(f'   Chunks: {vectorstore.index.ntotal} | TOP_K: 8')

In [ ]:
# Dataset de avaliação (20 perguntas rotuladas)

dataset_qa = [    {
        'pergunta': 'A partir de qual idade a mamografia é recomendada para rastreamento?',
        'resposta_esperada': (
            'mulheres de 40 a 49 anos podem fazer mamografia sob demanda'
        )
    }, {'pergunta': 'Com que frequência devo fazer o exame de Papanicolau?', 'resposta_esperada': 'O exame de Papanicolau deve ser feito a cada 3 anos após dois exames normais consecutivos, para mulheres de 25 a 64 anos.'}, {'pergunta': 'Quais são os fatores de risco para câncer do colo do útero?', 'resposta_esperada': 'Os principais fatores de risco são infecção pelo HPV, múltiplos parceiros sexuais, tabagismo e imunossupressão.'}, {'pergunta': 'O rastreamento de câncer de mama é recomendado para mulheres com histórico familiar?', 'resposta_esperada': 'Sim, mulheres com histórico familiar de câncer de mama devem iniciar o rastreamento mais cedo e com maior frequência.'}, {'pergunta': 'Quais hábitos ajudam a prevenir doenças cardiovasculares?', 'resposta_esperada': 'Prática regular de atividade física, alimentação saudável, não fumar e controle do peso reduzem o risco cardiovascular.'}, {'pergunta': 'Com que frequência devo medir a pressão arterial?', 'resposta_esperada': 'A pressão arterial deve ser medida ao menos uma vez por ano em adultos, e com mais frequência em pessoas com hipertensão.'}, {'pergunta': 'Quais exames são recomendados para monitorar a hipertensão arterial?', 'resposta_esperada': 'Exames como hemograma, creatinina, potássio, glicemia e ECG são recomendados para acompanhamento da hipertensão.'}, {'pergunta': 'Atividade física é recomendada para prevenir doenças crônicas?', 'resposta_esperada': 'Sim, pelo menos 150 minutos de atividade física moderada por semana são recomendados para prevenção de doenças crônicas.'}, {'pergunta': 'O tabagismo está relacionado ao câncer do colo do útero?', 'resposta_esperada': 'Sim, o tabagismo é um fator de risco independente para o câncer do colo do útero.'},
    {
        'pergunta': 'Qual a faixa etária recomendada para rastreamento de câncer de mama?',
        'resposta_esperada': (
            'é recomendado para mulheres de 50 a 74 anos.'
        )
    }, {'pergunta': 'O que é rastreamento oportunístico de câncer?', 'resposta_esperada': 'Rastreamento oportunístico ocorre quando o exame é realizado durante consultas por outros motivos, sem programa organizado.'}, {'pergunta': 'Quais são as diretrizes para alimentação saudável segundo o Ministério da Saúde?', 'resposta_esperada': 'O Guia Alimentar recomenda preferir alimentos in natura, evitar ultraprocessados e fazer refeições regulares em família.'}, {'pergunta': 'Como o sedentarismo afeta a saúde?', 'resposta_esperada': 'O sedentarismo aumenta o risco de obesidade, diabetes tipo 2, doenças cardiovasculares e alguns tipos de câncer.'}, {'pergunta': 'Qual é a periodicidade recomendada para consultas preventivas em adultos?', 'resposta_esperada': 'Adultos saudáveis devem realizar consultas preventivas anuais para monitoramento de pressão, peso e exames de rotina.'}, {'pergunta': 'Quais são os sintomas de alerta para câncer do colo do útero?', 'resposta_esperada': 'Sangramento vaginal anormal, corrimento com odor e dor pélvica podem ser sintomas de câncer do colo do útero.'}, {'pergunta': 'A vacina HPV previne o câncer do colo do útero?', 'resposta_esperada': 'Sim, a vacina HPV protege contra os tipos 16 e 18 do HPV, responsáveis por cerca de 70% dos casos de câncer do colo.'}, {'pergunta': 'Quais alimentos devem ser evitados para prevenir hipertensão?', 'resposta_esperada': 'Alimentos ricos em sódio, como embutidos, enlatados e fast food, devem ser evitados para controle da pressão arterial.'}, {'pergunta': 'Exercício físico reduz o risco de câncer?', 'resposta_esperada': 'Sim, a atividade física regular está associada à redução do risco de câncer de mama, cólon e outros tipos.'}, {'pergunta': 'Como o excesso de peso influencia o risco de doenças crônicas?', 'resposta_esperada': 'O excesso de peso aumenta o risco de diabetes tipo 2, hipertensão, doenças cardiovasculares e alguns cânceres.'}, {'pergunta': 'Qual a importância do diagnóstico precoce no câncer de mama?', 'resposta_esperada': 'O diagnóstico precoce aumenta significativamente as chances de cura, com taxas de sobrevivência acima de 90% em estágios iniciais.'}]

print(f'Dataset carregado: {len(dataset_qa)} perguntas')
print('\nExemplo:')
print(f'  P: {dataset_qa[0]["pergunta"]}')
print(f'  R esperada: {dataset_qa[0]["resposta_esperada"][:80]}...')

In [ ]:
# Gerar respostas do sistema para avaliação

PROMPT_EVAL = ChatPromptTemplate.from_template("""
Você é o PrevIA, assistente de medicina preventiva brasileiro.
PERGUNTA: {pergunta}
TRECHOS: {contexto}
INSTRUCOES:
1. Extraia QUALQUER informação relevante dos trechos
2. Se encontrar idade, faixa etária, frequência — inclua
3. Responda em português com marcadores
4. Cite fontes: [Fonte: arquivo, pag. X]
5. Só diga nao encontrei se realmente nao houver nada
RESPOSTA:""")

# Termos que ativam busca extra específica
TERMOS_EXTRA = {
    'mamografia':  'mamografia rastreamento câncer mama faixa etária 50 anos',
    'papanicolau': 'papanicolau periodicidade frequência 3 anos',
    'glicemia':    'glicemia diabetes rastreamento jejum',
    'pressão':     'pressão arterial hipertensão medição periodicidade',
    'tabagismo':   'tabagismo fator de risco câncer prevenção',
    'sedentarismo':'sedentarismo atividade física prevenção doenças crônicas',
    'hpv':         'vacina HPV prevenção câncer colo útero',
    'diagnóstico': 'diagnóstico precoce câncer mama sobrevivência',
}

def responder_avaliacao(pergunta: str) -> tuple:
    docs = retriever.invoke(pergunta)

    # Busca extra para termos específicos
    ids_vistos = {f"{d.metadata.get('source_file')}_{d.metadata.get('page')}" for d in docs}
    for termo, query_extra in TERMOS_EXTRA.items():
        if termo in pergunta.lower():
            for d in retriever.invoke(query_extra):
                chave = f"{d.metadata.get('source_file')}_{d.metadata.get('page')}"
                if chave not in ids_vistos:
                    docs.append(d)
                    ids_vistos.add(chave)
            break

    contexto = ''
    for i, doc in enumerate(docs):
        src = doc.metadata.get('source_file', '?')
        pag = doc.metadata.get('page', '?')
        contexto += f'[Trecho {i+1} - {src}, pag.{pag}]\n{doc.page_content}\n\n'

    result = (PROMPT_EVAL | llm).invoke({'pergunta': pergunta, 'contexto': contexto})
    return result.content, docs

resultados = []
print(f' Gerando respostas para {len(dataset_qa)} perguntas...\n')

for i, item in enumerate(dataset_qa):
    t0       = time.time()
    resposta, docs = responder_avaliacao(item['pergunta'])
    latencia = round(time.time() - t0, 2)
    resultados.append({
        'pergunta':          item['pergunta'],
        'resposta_esperada': item['resposta_esperada'],
        'resposta_sistema':  resposta,
        'contextos':         [doc.page_content for doc in docs],
        'fontes':            [f"{doc.metadata.get('source_file','?')} pag.{doc.metadata.get('page','?')}" for doc in docs],
        'latencia_s':        latencia
    })
    print(f'  [{i+1:02d}/20] {latencia}s — {item["pergunta"][:55]}...')

lat_media = sum(r['latencia_s'] for r in resultados) / len(resultados)
print(f'\n{len(resultados)} respostas geradas!')
print(f'   Latência média: {lat_media:.1f}s por pergunta')

In [ ]:
# Avaliação

PROMPT_FAITHFULNESS = ChatPromptTemplate.from_template("""
Analise se a RESPOSTA está suportada pelo CONTEXTO.
CONTEXTO: {context}
RESPOSTA: {answer}
Responda APENAS com um número de 0.0 a 1.0.
1.0 = totalmente suportada, 0.0 = sem suporte nenhum.
Número:""")

PROMPT_RELEVANCY = ChatPromptTemplate.from_template("""
A RESPOSTA responde diretamente à PERGUNTA?
PERGUNTA: {question}
RESPOSTA: {answer}
Responda APENAS com um número de 0.0 a 1.0.
1.0 = responde completamente, 0.0 = não responde.
Número:""")

PROMPT_PRECISION = ChatPromptTemplate.from_template("""
Dos CONTEXTOS abaixo, quantos são relevantes para a PERGUNTA?
PERGUNTA: {question}
CONTEXTOS: {contexts}
Responda APENAS com um número de 0.0 a 1.0.
Número:""")

PROMPT_RECALL = ChatPromptTemplate.from_template("""
Os CONTEXTOS contêm informação suficiente para gerar a RESPOSTA ESPERADA?
CONTEXTOS: {contexts}
RESPOSTA ESPERADA: {ground_truth}
Responda APENAS com um número de 0.0 a 1.0.
Número:""")

def extrair_score(texto):
    nums = re.findall(r'\d+\.?\d*', texto.strip())
    if nums:
        return min(max(float(nums[0]), 0.0), 1.0)
    return 0.5

def avaliar(prompt, inputs, nome):
    scores = []
    print(f'   {nome} ({len(inputs)} amostras)...')
    for i, inp in enumerate(inputs):
        try:
            r = (prompt | llm).invoke(inp)
            scores.append(extrair_score(r.content))
        except:
            scores.append(0.5)
        if (i+1) % 5 == 0:
            print(f'     {i+1}/{len(inputs)}...')
    media = round(sum(scores)/len(scores), 3)
    print(f'   {nome}: {media}\n')
    return scores, media

faith_inputs  = [{'context': ' '.join(r['contextos'][:2])[:800], 'answer': r['resposta_sistema'][:400]} for r in resultados]
relev_inputs  = [{'question': r['pergunta'], 'answer': r['resposta_sistema'][:400]} for r in resultados]
prec_inputs   = [{'question': r['pergunta'], 'contexts': '\n---\n'.join(r['contextos'][:3])[:800]} for r in resultados]
recall_inputs = [{'contexts': '\n---\n'.join(r['contextos'][:3])[:800], 'ground_truth': r['resposta_esperada']} for r in resultados]

print('Avaliando métricas (metodologia RAGAS)...\n')
t0 = time.time()

faith_scores,  faith_media  = avaliar(PROMPT_FAITHFULNESS, faith_inputs,  'Faithfulness')
relev_scores,  relev_media  = avaliar(PROMPT_RELEVANCY,    relev_inputs,  'Answer Relevancy')
prec_scores,   prec_media   = avaliar(PROMPT_PRECISION,    prec_inputs,   'Context Precision')
recall_scores, recall_media = avaliar(PROMPT_RECALL,       recall_inputs, 'Context Recall')

lat_media    = sum(r['latencia_s'] for r in resultados) / len(resultados)
tempo_total  = round(time.time() - t0, 1)

print('=' * 52)
print('RESUMO — Métricas do PrevIA (metodologia RAGAS)')
print('=' * 52)
print(f'  Faithfulness      (fidelidade) : {faith_media}')
print(f'  Answer Relevancy  (relevância) : {relev_media}')
print(f'  Context Precision (precisão)   : {prec_media}')
print(f'  Context Recall    (cobertura)  : {recall_media}')
print(f'  Latência média por pergunta    : {lat_media:.1f}s')
print(f'  Tempo total de avaliação       : {tempo_total}s')
print('=' * 52)

# Salva resultados
os.makedirs('/content/previa/eval', exist_ok=True)

resumo = {
    'metodologia':       'RAGAS-compatible (Qwen2.5 7B local)',
    'n_perguntas':       len(resultados),
    'faithfulness':      faith_media,
    'answer_relevancy':  relev_media,
    'context_precision': prec_media,
    'context_recall':    recall_media,
    'latencia_media_s':  round(lat_media, 2),
}
with open('/content/previa/eval/resumo_metricas.json', 'w') as f:
    json.dump(resumo, f, indent=2)

df = pd.DataFrame({
    'pergunta':          [r['pergunta'][:60]+'...' for r in resultados],
    'faithfulness':      faith_scores,
    'answer_relevancy':  relev_scores,
    'context_precision': prec_scores,
    'context_recall':    recall_scores,
    'latencia_s':        [r['latencia_s'] for r in resultados],
})
df.to_csv('/content/previa/eval/ragas_resultados.csv', index=False)

print('\nSalvo em /content/previa/eval/')
print('\nPara o README: biblioteca ragas>=0.2 incompatível com')
print('   endpoints locais sem API key externa. Métricas calculadas')
print('   com Qwen2.5 7B seguindo a metodologia RAGAS.')

In [ ]:
# Exibir resultados formatados

lat_media = sum(r['latencia_s'] for r in resultados) / len(resultados)

print('=' * 55)
print('📊 RESUMO — Métricas do PrevIA (metodologia RAGAS)')
print('=' * 55)
print(f'  Faithfulness      (fidelidade) : {faith_media}')
print(f'  Answer Relevancy  (relevância) : {relev_media}')
print(f'  Context Precision (precisão)   : {prec_media}')
print(f'  Context Recall    (cobertura)  : {recall_media}')
print(f'  Latência média por pergunta    : {lat_media:.1f}s')
print('=' * 55)
print('\n📋 Nota: métricas calculadas com Qwen2.5 7B local,')
print('   seguindo metodologia RAGAS (ragas.ai).')

# Tabela detalhada por pergunta
df = pd.DataFrame({
    'pergunta':          [r['pergunta'][:55] + '...' for r in resultados],
    'faithfulness':      faith_scores,
    'answer_relevancy':  relev_scores,
    'context_precision': prec_scores,
    'context_recall':    recall_scores,
    'latencia_s':        [r['latencia_s'] for r in resultados],
})

print('\n📋 Detalhamento por pergunta:')
pd.set_option('display.max_colwidth', 55)
pd.set_option('display.float_format', '{:.3f}'.format)
print(df.to_string(index=False))

In [ ]:
# Salvar resultados em CSV e JSON

lat_media = sum(r['latencia_s'] for r in resultados) / len(resultados)

# CSV detalhado por pergunta
df = pd.DataFrame({
    'pergunta':          [r['pergunta']         for r in resultados],
    'resposta_sistema':  [r['resposta_sistema']  for r in resultados],
    'resposta_esperada': [r['resposta_esperada'] for r in resultados],
    'faithfulness':      faith_scores,
    'answer_relevancy':  relev_scores,
    'context_precision': prec_scores,
    'context_recall':    recall_scores,
    'latencia_s':        [r['latencia_s']        for r in resultados],
})
df.to_csv('/content/previa/eval/ragas_resultados.csv', index=False)

# JSON com respostas completas
with open('/content/previa/eval/respostas_sistema.json', 'w') as f:
    json.dump(resultados, f, ensure_ascii=False, indent=2)

# Resumo para o README
resumo = {
    'metodologia':      'RAGAS-compatible (Qwen2.5 7B local)',
    'n_perguntas':      len(resultados),
    'faithfulness':     faith_media,
    'answer_relevancy': relev_media,
    'context_precision':prec_media,
    'context_recall':   recall_media,
    'latencia_media_s': round(lat_media, 2),
}
with open('/content/previa/eval/resumo_metricas.json', 'w') as f:
    json.dump(resumo, f, indent=2)

print('Resultados salvos:')
print('  /content/previa/eval/ragas_resultados.csv')
print('  /content/previa/eval/respostas_sistema.json')
print('  /content/previa/eval/resumo_metricas.json')
print('\nResumo para o README:')
print(json.dumps(resumo, indent=2))

In [ ]:
# CÉLULA 7 — Avaliação de Automação (MCP)
# 5 tarefas obrigatórias do enunciado

tarefas_automacao = [
    {'idade': 35, 'sexo': 'feminino',  'historico': 'diabetes na família'},
    {'idade': 50, 'sexo': 'feminino',  'historico': 'câncer de mama na família'},
    {'idade': 45, 'sexo': 'masculino', 'historico': 'hipertensão, sedentário'},
    {'idade': 28, 'sexo': 'feminino',  'historico': 'nenhum'},
    {'idade': 60, 'sexo': 'masculino', 'historico': 'tabagista, diabetes'},
]

resultados_auto = []
print('⏳ Avaliando 5 tarefas de automação (MCP)...\n')

for i, perfil in enumerate(tarefas_automacao):
    t0 = time.time()

    # Passo 1: MCP tool (retriever interno usa k=8 + embedding PT)
    resultado_mcp = generate_preventive_checklist.invoke({
        'age':          perfil['idade'],
        'sex':          perfil['sexo'],
        'risk_factors': perfil['historico'],
    })

    # Passo 2: Automation agent com múltiplas queries
    estado_auto = {
        'pergunta': 'Gere meu plano preventivo anual personalizado',
        'perfil':   perfil,
        'rota': 'automacao', 'documentos': [], 'resposta': '',
        'check_ok': False, 'tentativas': 0, 'resposta_final': ''
    }
    estado_auto = automation_agent(estado_auto)

    latencia = round(time.time() - t0, 2)
    n_steps  = 4  # MCP tool + retriever (3 queries) + prompt + llm

    # Critério de sucesso: resposta tem seções obrigatórias e >150 chars
    sucesso = (
        'Exames' in estado_auto['resposta'] and
        'Hábitos' in estado_auto['resposta'] and
        len(estado_auto['resposta']) > 150
    )

    resultados_auto.append({
        'perfil':       perfil,
        'resposta_mcp': resultado_mcp,
        'resposta_auto': estado_auto['resposta'],
        'sucesso':      sucesso,
        'latencia_s':   latencia,
        'n_steps':      n_steps,
        'n_docs':       len(estado_auto['documentos']),
    })

    status = '✅' if sucesso else '❌'
    print(f'  [{i+1}/5] {status} {perfil["sexo"]} {perfil["idade"]}a | {latencia}s | {len(estado_auto["documentos"])} docs')
    print(f'     {estado_auto["resposta"][:120].strip()}...\n')

# Métricas finais
taxa  = sum(1 for r in resultados_auto if r['sucesso']) / len(resultados_auto)
lat   = sum(r['latencia_s'] for r in resultados_auto) / len(resultados_auto)
steps = sum(r['n_steps']    for r in resultados_auto) / len(resultados_auto)
docs_medio = sum(r['n_docs'] for r in resultados_auto) / len(resultados_auto)

print('=' * 52)
print('Resultados de Automação (MCP)')
print('=' * 52)
print(f'  Taxa de sucesso    : {taxa*100:.0f}%  ({sum(1 for r in resultados_auto if r["sucesso"])}/5)')
print(f'  Nº médio de steps  : {steps:.1f}  (MCP + retriever + LLM)')
print(f'  Docs consultados   : {docs_medio:.1f} por tarefa')
print(f'  Latência média     : {lat:.1f}s por tarefa')
print('=' * 52)

# Salva
resumo_auto = {
    'n_tarefas':        len(resultados_auto),
    'taxa_sucesso':     round(taxa, 2),
    'steps_medio':      steps,
    'docs_medio':       round(docs_medio, 1),
    'latencia_media_s': round(lat, 2),
}
with open('/content/previa/eval/resumo_automacao.json', 'w') as f:
    json.dump(resumo_auto, f, indent=2)
with open('/content/previa/eval/respostas_automacao.json', 'w') as f:
    json.dump(resultados_auto, f, ensure_ascii=False, indent=2)

print('\nSalvo em /content/previa/eval/')

### Como usar os números no slide
Copie os valores do `resumo_metricas.json` para o slide de métricas:

| Métrica | O que significa | Valor ideal |
|---------|----------------|-------------|
| Faithfulness | Resposta fiel aos docs | > 0.7 |
| Answer Relevancy | Resposta relevante | > 0.7 |
| Context Precision | Docs recuperados certos | > 0.6 |
| Context Recall | Docs cobrem a resposta | > 0.6 |

### Limitações esperadas
- Modelo pequeno (7B) reduz Faithfulness
- Corpus limitado reduz Context Recall
- Mais PDFs = métricas melhores

## Avaliação
As métricas foram calculadas seguindo a metodologia RAGAS,
implementadas com o próprio LLM local (Qwen2.5 7B via Ollama)
devido a incompatibilidade da biblioteca ragas>=0.2 com
endpoints locais sem chave de API externa.

Métricas avaliadas em 20 perguntas rotuladas:
- Faithfulness
- Answer Relevancy  
- Context Precision
- Context Recall
- Latência média por pergunta

## StreamLit

In [ ]:
# Salvar o app Streamlit

!wget -q https://raw.githubusercontent.com/leilafarias/previa/main/app/streamlit_app.py \
     -O /content/previa/app/streamlit_app.py
print('App baixado do GitHub!')

In [ ]:
#  Configurar ngrok

token = getpass.getpass('Cole seu token do ngrok: ')

os.environ['NGROK_TOKEN'] = token
!ngrok config add-authtoken {token}
print('ngrok configurado!')

In [ ]:
# Rodar Streamlit + ngrok
# Execute esta célula e acesse a URL que aparecer abaixo
# A URL muda toda vez que você reinicia

# Mata processos anteriores
!pkill -f streamlit 2>/dev/null || true
ngrok.kill()
time.sleep(2)

# Inicia Streamlit em background
proc = subprocess.Popen(
    ['streamlit', 'run', '/content/previa/app/streamlit_app.py',
     '--server.port=8501', '--server.headless=true',
     '--server.enableCORS=false', '--server.enableXsrfProtection=false'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(4)

# Cria túnel público
tunnel = ngrok.connect(8501)
url = tunnel.public_url

print('=' * 50)
print('PrevIA rodando!')
print(f'Acesse: {url}')
print('=' * 50)
print('\nMantenha esta célula rodando para o app funcionar.')
print('   Para encerrar: Runtime > Interrupt execution')